# Maps of Taunton and Worcester area

## Code setup

In [1]:
import os
import geopandas
import numpy as np
import pandas as pd
import copy

# Plotting:
import contextily as ctx  # map tiles
import matplotlib.pyplot as plt
import cmasher as cmr  # for additional colormaps

In [2]:
dir_data = 'data'
dir_output = 'output'
dir_images = 'images'
# Scenario results:
file_scen_params = 'scenario_parameters.csv'
file_results_comparison = 'results_full_comparison_taunworc_base.csv'
# Geometry:
file_england_lsoa = 'Lower_layer_super_output_areas_(E+W)_2011_Boundaries_(Generalised_Clipped)_V2.zip'
file_uk_outlines = 'Countries_(December_2022)_GB_BFC.zip'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'

## Load data

In [3]:
p = os.path.join(dir_output, file_results_comparison)
comparison = pd.read_csv(p, index_col='LSOA')

In [4]:
print(comparison.columns)

Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'transfer_unit_base', 'transfer_required_base',
       'drip_ship_mt_travel_time_base', 'admissions_mt_base',
       'transfers_mt_base', 'admissions_mt_transferloss0!6_base',
       'transfers_mt_transferloss0!6_base',
       'admissions_weighted_drip_ship_mrs_0-2_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!6_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base',
       'drip_ship_mrs_0-2_base', 'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_mrs_0-2_transferloss0!6_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base',
       'nearest_ivt_unit_taunworc', 'transfer_unit_taunworc',
       'transfer_required_taunworc', 'drip_ship_mt_travel_time_taunworc',
       'admissions_mt_taunworc', 'transfers_mt_taunworc',
       'admissions_mt_transferloss0!6_taunworc',
       'transfer

Limit to LSOA that have changed with the opening of a Worcester or Taunton unit:

In [5]:
comparison = comparison[comparison['changed']]

In [6]:
used_hospitals = sorted(list(set(comparison[[
    'transfer_unit_base', 'transfer_unit_taunworc',
    'nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc'
]].dropna().values.flatten())))

# Get all postcodes from comparisonMT units
comparitor_mt_postcodes = (
    comparison['transfer_unit_taunworc'].unique().tolist())

print(used_hospitals)
print(comparitor_mt_postcodes)

['B152TH', 'BS105NB', 'DT12JY', 'DY12HQ', 'EX25DW', 'EX314JB', 'GL537AN', 'HR12ER', 'PL68DH', 'SO166YD', 'TA15DA', 'WR51DD']
['WR51DD', 'TA15DA']


## Geometry

In [7]:
# Load the England and Wales LSOA shapefile
filename = os.path.join(dir_data, file_england_lsoa)
lsoa_gdf = geopandas.read_file(filename)

# Set or convert to British National Grid (EPSG:27700)
if lsoa_gdf.crs is None:
    lsoa_gdf = lsoa_gdf.set_crs("EPSG:27700")
else:
    lsoa_gdf = lsoa_gdf.to_crs("EPSG:27700")

In [8]:
# Get the England and Wales outline
filename = os.path.join(dir_data, file_uk_outlines)
countries_gdf = geopandas.read_file(filename)

# Set or convert to British National Grid (EPSG:27700)
if countries_gdf.crs is None:
    countries_gdf = countries_gdf.set_crs("EPSG:27700")
else:
    countries_gdf = countries_gdf.to_crs("EPSG:27700")

# Only keep England and Wales outlines:
mask = countries_gdf["CTRY22NM"].isin(["England", "Wales"])
countries_gdf = countries_gdf.loc[mask]
england = countries_gdf.boundary

In [9]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
hospitals_df = pd.read_csv(p)

hospitals_gdf = geopandas.GeoDataFrame(
    hospitals_df,
    geometry=geopandas.points_from_xy(
        pd.to_numeric(hospitals_df["long"]),
        pd.to_numeric(hospitals_df["lat"]),
    ),
    crs="EPSG:4326",
).to_crs("EPSG:27700")

# Only keep hospitals in England:
hospitals_gdf = hospitals_gdf.loc[hospitals_gdf["Country"] == "England"]
# Limit hospitals_gdf to only those used in the comparison
hospitals_gdf = hospitals_gdf.loc[hospitals_gdf['Postcode'].isin(used_hospitals)]

# Find comparitor_mt_postcodes not labelled as Use_MT = 1 in the hospitals_gdf
extra_mt_postcodes = list(
    set(hospitals_gdf[hospitals_gdf['Use_MT'] == 0]['Postcode'].tolist()) &
    set(comparitor_mt_postcodes)
)

# Create an extra column in the hospitals_gdf to indicate if a hospital is an extra MT unit
hospitals_gdf['extra_MT_unit'] = (
    hospitals_gdf['Postcode'].isin(extra_mt_postcodes).astype(int))

display(hospitals_gdf.head(3))

,SSNAP name,Postcode,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT,geometry,extra_MT_unit
71,Russells Hall Hospital,DY12HQ,England,North Midlands,392052,289456,-2.118518,52.502944,Dudley,E01009884,1,0,POINT (392052.001 289456),0
80,Derriford Hospital,PL68DH,England,SW Peninsula,249930,59564,-4.113684,50.416720,Plymouth,E01015092,1,1,POINT (249930.001 59563.999),0
81,North Devon District Hospital,EX314JB,England,SW Peninsula,256506,134540,-4.050431,51.092169,Barnstaple,E01020134,1,0,POINT (256506.001 134539.999),0


## Setup for map

In [10]:
# Merge the results with the LSOA shapefile
comparison = pd.merge(lsoa_gdf, comparison, left_on='LSOA11NM',
                      right_on='area_lower', how='right')

In [11]:
# Get limits of geometry for plotting
bounds = comparison.geometry.bounds
min_lng = bounds['minx'].min()
max_lng = bounds['maxx'].max()
min_lat = bounds['miny'].min()
max_lat = bounds['maxy'].max()

# Include the locations of all base and comparator MT hospitals
hospital_points = hospitals_gdf.loc[
    hospitals_gdf['Postcode'].isin(used_hospitals), 'geometry'
]
hospital_lngs = hospital_points.x
hospital_lats = hospital_points.y

min_lng = min(min_lng, hospital_lngs.min())
max_lng = max(max_lng, hospital_lngs.max())
min_lat = min(min_lat, hospital_lats.min())
max_lat = max(max_lat, hospital_lats.max())

# Add a small buffer to the limits for better visualization
buffer = 10000  # metres
min_lng -= buffer
max_lng += buffer * 4
min_lat -= buffer
max_lat += buffer


# Manually remove Hospital_City for Dudley if in used_hopsital list
if 'DY12HQ' in used_hospitals:
    # Remove row
    hospitals_gdf = hospitals_gdf[hospitals_gdf['Postcode'] != 'DY12HQ']

## Draw map

Helper functions:

In [12]:
def get_color_limits(series, vmin=None, vmax=None):
    """
    Return finite plotting limits.

    Any limit passed explicitly is used as-is; any limit left as None is
    taken from the data. Guards against a zero-width or inverted range.
    """
    values = pd.to_numeric(series, errors="coerce").dropna()

    if vmin is None:
        vmin = values.min() if not values.empty else 0
    if vmax is None:
        vmax = values.max() if not values.empty else vmin + 1

    if vmax <= vmin:
        # Keep whichever limit the caller fixed and move the other one
        if vmax is not None and vmin == values.min():
            vmin = vmax - 1
        else:
            vmax = vmin + 1

    return vmin, vmax

In [13]:
def plot_map(ax, data, **kwargs):
    """
    Plot a choropleth and its colorbar.

    vmin / vmax set the colour scale limits. Leave either as None to set
    it automatically from the data.
    """
    data.plot(ax=ax, **kwargs)

In [14]:
def get_hospital(hospitals_gdf, postcode):
    """Return the first matching hospital, or None if it is unavailable."""
    matches = hospitals_gdf.loc[hospitals_gdf["Postcode"].eq(postcode)]
    return None if matches.empty else matches.iloc[0]

In [15]:
def draw_arrow(ax, hospitals_gdf, start_postcode, end_postcode):
    """Draw an arrow only when both hospital locations are available."""
    start = get_hospital(hospitals_gdf, start_postcode)
    end = get_hospital(hospitals_gdf, end_postcode)

    if start is None or end is None:
        return

    ax.annotate(
        "",
        xy=(end.geometry.x, end.geometry.y),
        xytext=(start.geometry.x, start.geometry.y),
        arrowprops={
            "arrowstyle": "simple",
            "facecolor": "white",
            "lw": 0.7,
            "mutation_scale": 30,
        },
        zorder=3,
    )

In [16]:
def plot_hospitals(ax, hospitals_gdf, show_new_mt=False):
    """Plot hospital markers and labels."""
    ivt_mask = pd.to_numeric(
        hospitals_gdf["Use_IVT"], errors="coerce"
    ).eq(1)
    mt_mask = pd.to_numeric(
        hospitals_gdf["Use_MT"], errors="coerce"
    ).eq(1)
    new_mt_mask = pd.to_numeric(
        hospitals_gdf["extra_MT_unit"], errors="coerce"
    ).eq(1)

    hospitals_gdf.loc[ivt_mask].plot(
        ax=ax,
        edgecolor="k",
        facecolor="w",
        markersize=30,
        marker="o",
        zorder=99,
    )
    hospitals_gdf.loc[mt_mask].plot(
        ax=ax,
        edgecolor="k",
        facecolor="w",
        markersize=200,
        marker="*",
        zorder=99,
    )

    if show_new_mt:
        hospitals_gdf.loc[new_mt_mask].plot(
            ax=ax,
            edgecolor="k",
            facecolor="r",
            markersize=300,
            marker="*",
            zorder=99,
        )

    ivt_only_mask = ivt_mask & ~mt_mask
    if show_new_mt:
        ivt_only_mask &= ~new_mt_mask

    label_groups = [
        (ivt_only_mask, 10, "black"),
        (mt_mask, 12, "red"),
    ]
    if show_new_mt:
        label_groups.append((new_mt_mask, 12, "red"))

    for mask, fontsize, color in label_groups:
        for _, hospital in hospitals_gdf.loc[mask].iterrows():
            ax.annotate(
                hospital["Hospital_City"],
                xy=(hospital.geometry.x, hospital.geometry.y),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=fontsize,
                color=color,
                zorder=100,
                bbox={
                    "facecolor": "white",
                    "edgecolor": "none",
                    "alpha": 0.8,
                    "pad": 1,
                },
            )

In [17]:
def add_basemap(ax):
    ctx.add_basemap(
        ax,
        source="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
        crs="EPSG:27700",
        zoom="auto",
        alpha=0.25,
        zorder=1,
    )

Make the plots:

In [18]:
def main_plot(data, hospitals_gdf, map_dict, savename='', show=True):
    # Setup for arrows:
    base_combinations = (
        data[['nearest_ivt_unit_base', 'transfer_unit_base']]
        .drop_duplicates()
    )
    comparator_combinations = (
        data[['nearest_ivt_unit_taunworc', 'transfer_unit_taunworc']]
        .drop_duplicates()
    )

    fig, axs = plt.subplots(1, len(map_dict), figsize=(20, 6))


    # Plot axis-specific data:
    for ax, ax_dict in zip(axs, map_dict.values()):
        # Plot main map data.
        plot_map(ax, data, **ax_dict['lsoa'])
        # Hospital markers:
        plot_hospitals(ax, hospitals_gdf,
                       show_new_mt=ax_dict['markers_show_new_mt'])
        # Draw arrows:
        if ax_dict['arrows'] == 'base':
            for _, row in base_combinations.iterrows():
                draw_arrow(ax, hospitals_gdf, row['nearest_ivt_unit_base'],
                           row['transfer_unit_base'])
        else:
            for _, row in comparator_combinations.iterrows():
                destination = get_hospital(hospitals_gdf,
                                           row['transfer_unit_taunworc'])
                if (destination is not None and
                    destination["extra_MT_unit"] == 1):
                    draw_arrow(ax, hospitals_gdf,
                               row['nearest_ivt_unit_taunworc'],
                               row['transfer_unit_taunworc'])
        # Axis title:
        ax.set_title(ax_dict['ax_title'], fontsize=12)
    
    # Basemaps:
    for ax in axs:
        add_basemap(ax)

    # England outline:
    for ax in axs:
        england.plot(ax=ax, edgecolor="k", facecolor="#00000000", linewidth=1,
                     zorder=0)

    
    # Figure setup:    
    for ax in axs:
        ax.set_xlim(min_lng, max_lng)
        ax.set_ylim(min_lat, max_lat)
        ax.apply_aspect()
        ax.set_axis_off()
    fig.tight_layout()

    # Save and/or display picture:
    if len(savename) > 0:
        fig.savefig(p, dpi=300, bbox_inches="tight")
    else:
        pass
    if show:
        plt.show()
    else:
        plt.close()

Setup:

In [19]:
data = comparison.copy(deep=True)
data['change_in_lvo_drip_ship_ivt_mt_mrs_0-2'] *= 100
data['change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6'] *= 100
data['change_in_drip_ship_mt_travel_time'] *= -1

In [20]:
cols_outcome = ['change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
                'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6']
vmin_outcomes = data[cols_outcome].min(axis=None)
vmax_outcomes = data[cols_outcome].max(axis=None)

print(vmin_outcomes, vmax_outcomes)

0.0 8.7


In [21]:
map_dict = {
    'time_to_mt_unit': {
        'lsoa': dict(
            column='drip_ship_mt_travel_time_base',
            cmap="afmhot_r",
            missing_kwds={"color": "lightgrey"},
            antialiased=False,
            linewidth=0,
            legend=True,
            legend_kwds=dict(
                shrink=0.5,
                pad=0.01,
                fraction=0.05,
                label=("Travel time to MT unit "
                       "(minutes, including non-travel delay)"),
            ),
        ),
        'markers_show_new_mt': False,
        'arrows': 'base',
        'ax_title': "Travel time to MT unit (including non-travel delay)"
    },
    'change_in_time_to_mt_unit': {
        'lsoa': dict(
            column='change_in_drip_ship_mt_travel_time',
            cmap="cmr.pepper",
            missing_kwds={"color": "lightgrey"},
            antialiased=False,
            linewidth=0,
            vmin=0,
            legend=True,
            legend_kwds=dict(
                shrink=0.5,
                pad=0.01,
                fraction=0.05,
                label=("Improvement in travel time to MT unit "
                       "(minutes, including non-travel delay)"),
            ),
        ),
        'markers_show_new_mt': True,
        'arrows': 'changed',
        'ax_title': ("Improvement in travel time to MT unit "
                     "(including non-travel delay,\ncomparator - base)")
    },
    'outcome_improvement': {
        'lsoa': dict(
            column='change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
            cmap="inferno",
            missing_kwds={"color": "lightgrey"},
            antialiased=False,
            linewidth=0,
            vmin=vmin_outcomes,
            vmax=vmax_outcomes,
            legend=True,
            legend_kwds=dict(
                shrink=0.5,
                pad=0.01,
                fraction=0.05,
                label=("Improvement (percentage points) in mRS 0-2 outcomes\n"
                       "for treated LVO (comparator - base)"),
            ),
        ),
        'markers_show_new_mt': True,
        'arrows': 'changed',
        'ax_title': ("Improvement (percentage points) in mRS 0-2 outcomes\n"
                     "for treated LVO (comparator - base)")
    },
}

map_dict_lossy = copy.deepcopy(map_dict)
map_dict_lossy['outcome_improvement']['lsoa']['column'] = (
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6')

In [22]:
print(map_dict['outcome_improvement']['lsoa']['column'])
print(map_dict_lossy['outcome_improvement']['lsoa']['column'])

change_in_lvo_drip_ship_ivt_mt_mrs_0-2
change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6


In [23]:
output_name = 'taunworc'
file_maps = f'comparison_map_three_in_one_{output_name}.png'
p = os.path.join(dir_images, file_maps)

main_plot(data, hospitals_gdf, map_dict, savename=p, show=False)

In [24]:
output_name = 'taunworc_transferloss0!6'
file_maps = f'comparison_map_three_in_one_{output_name}.png'
p = os.path.join(dir_images, file_maps)

main_plot(data, hospitals_gdf, map_dict_lossy, savename=p, show=False)